# Ejecutivos desde reporting
Generador independiente de `archivados/bases_plantillas/archivos/reporte_ejecutivos.xlsx`, con la hoja **Ejecutivos** y la tabla **t_ejecutivos**. Lee las 37 columnas de `reporting.datos_ejecutivos` y verifica sus tipos SQL.

El período inicial es **17/08/2026 a 04/10/2026**, inclusive, ordenado por unidad de planificación, RUT y fecha. Son datos diarios: no se crean intervalos adicionales.

`tiempo_*` y `Aux objetivo` conservan las **horas decimales** de la vista: 42 horas siguen siendo 42. `tmo_seg` conserva segundos y `ocupación_gen` su escala original. Se respeta la precisión de origen; los nulos quedan vacíos y los identificadores se escriben como texto.

Se informan claves nulas o repetidas sin eliminar filas. La conexión se cierra antes de escribir Excel; todos los valores exportados se validan antes de sustituir el destino.

In [1]:
from datetime import date
from pathlib import Path
import polars as pl
from IPython.display import display
from analytics.excel_reportes import TablaDatos, generar_libro

actual = Path.cwd().resolve()
BASE_DIR = next((p for p in (actual, *actual.parents)
                 if (p / "db_conn.py").is_file() and (p / "pyproject.toml").is_file()), None)
if BASE_DIR is None:
    raise FileNotFoundError("Abrir el notebook dentro del proyecto.")
import sys
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from analytics.datos_reportes import cargar_ejecutivos, formatos_ejecutivos
DESDE = date(2026, 8, 17)
HASTA = date(2026, 10, 4)
RUTA_EXCEL = BASE_DIR / "archivados" / "bases_plantillas" / "archivos" / "reporte_ejecutivos.xlsx"

In [2]:
print("Consultando Ejecutivos...")
df = cargar_ejecutivos(DESDE, HASTA, BASE_DIR)
claves = ["unidad_planificación", "rut", "fecha"]
print(f"Ejecutivos: {df.height:,} filas; {df.width} columnas.")
print("Filas con clave repetida:", int(df.select(claves).is_duplicated().sum()))
print("Filas con clave nula:", df.filter(pl.any_horizontal(pl.col(claves).is_null())).height)
display(pl.DataFrame({"columna": list(df.schema), "tipo": [str(t) for t in df.schema.values()]}))

Consultando Ejecutivos...


Ejecutivos: 24,633 filas; 37 columnas.
Filas con clave repetida: 0
Filas con clave nula: 0


columna,tipo
str,str
"""unidad_planificación""","""String"""
"""rut""","""String"""
"""identificador""","""String"""
"""fecha""","""Date"""
"""texto_turno""","""String"""
…,…
"""ausencia_de_conexion""","""Int64"""
"""fecha (índice de meses)""","""Int64"""
"""fecha (mes)""","""String"""


In [3]:
print("Escribiendo y validando el Excel...")
generar_libro([TablaDatos("t_ejecutivos", "Ejecutivos", df, formatos_ejecutivos())], RUTA_EXCEL)
print(f"Generado: {RUTA_EXCEL.name}; {df.height:,} filas.")

Escribiendo y validando el Excel...


Generado: reporte_ejecutivos.xlsx; 24,633 filas.
